# Predict NF-v3 flows with a trained model

Loads a model bundle and predicts any CSV with the 47 NF-v3 features (extra columns are ignored). No dataset identity is used for prediction. Optionally scores the predictions against the CSV's `Attack` column, mapped with the label map of the dataset named in `LABELS_FROM` (used only for that mapping).

* **Runtime:** minutes
* **GPU:** not needed
* **Before running:** follow [docs/DATA.md](https://github.com/selimsidan/dmoe_nids/blob/main/docs/DATA.md) to put the four NF-v3 CSVs in your Drive (or set `DATA_ROOT`).
* **How to use:** edit only the SETTINGS cell, then *Runtime > Run all*. Rerunning after a disconnect continues where training stopped.


In [ ]:
# ============== SETTINGS: the only cell you edit ==============
REPO_REF    = "main"     # branch or release tag (e.g. "v1.0.0") of the repository
STORAGE     = "drive"    # "drive": Google Drive (persists) | "local": /content (lost at disconnect)
DATA_ROOT   = None       # None: Colab secret NIDS_DATA_DIR, else MyDrive/NIDS_datasets
OUTPUT_ROOT = None       # None: Colab secret NIDS_OUTPUT_DIR, else MyDrive/NIDS_outputs
# Experiment
BUNDLE      = "hf:recommended"   # "hf:recommended", "hf:<condition>/seed<k>", or a bundle folder path
INPUT_CSV   = None   # path to a CSV with the NF-v3 feature columns
OUTPUT_CSV  = None   # None: <INPUT_CSV without .csv>_predictions.csv
WITH_PROBS  = False  # one probability column per class
WITH_GATE   = False  # gate weights and the top expert (MoE models)
LABELS_FROM = None   # e.g. "NF-ToN-IoT-v3" to also evaluate against the Attack column
# ===============================================================

In [ ]:
# Setup: clone the repository, install it, mount storage and resolve the roots.
import os, subprocess, sys

REPO_URL, TARGET = "https://github.com/selimsidan/dmoe_nids.git", "/content/dmoe_nids"

def _git(*args, cwd=None, env=None):
    subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True,
                   env={**os.environ, **(env or {})})

def _fetch(env=None):
    if os.path.isdir(os.path.join(TARGET, ".git")):
        _git("fetch", "--quiet", "--tags", "origin", cwd=TARGET, env=env)
    else:
        _git("clone", "--quiet", REPO_URL, TARGET, env=env)

try:
    _fetch()
except subprocess.CalledProcessError:
    # Only for a private repository or fork: retry with the Colab secret GITHUB_TOKEN (never printed).
    import base64
    from google.colab import userdata
    _token = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    _fetch({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {_token}"})
    del _token
_git("checkout", "--quiet", REPO_REF, cwd=TARGET)
if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=TARGET).returncode == 0:
    # Fast-forward to the commits fetched above (no second download, so no token needed).
    _git("merge", "--quiet", "--ff-only", f"origin/{REPO_REF}", cwd=TARGET)

# Installs the package and any missing dependencies; Colab's own torch build is kept.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", TARGET], check=True)
sys.path.insert(0, os.path.join(TARGET, "src"))
os.environ["PYTHONPATH"] = os.path.join(TARGET, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(TARGET)

import dmoe.colab
ROOTS = dmoe.colab.setup(STORAGE, DATA_ROOT, OUTPUT_ROOT, require_data=False)

In [ ]:
import os, pandas as pd, torch
from IPython.display import display
from dmoe.evaluation.evaluate import evaluate
from dmoe.inference.bundle import load_bundle

assert INPUT_CSV, 'set INPUT_CSV in the SETTINGS cell'
bundle = load_bundle(BUNDLE, 'cuda' if torch.cuda.is_available() else 'cpu')
print('bundle:', bundle.metadata.get('condition'), 'seed', bundle.metadata.get('seed'),
      'test macro-F1', round(bundle.metadata.get('test_macro_f1', float('nan')), 4))
output = OUTPUT_CSV or os.path.splitext(INPUT_CSV)[0] + '_predictions.csv'
first = True
for chunk in pd.read_csv(INPUT_CSV, chunksize=262_144, low_memory=False):
    bundle.predict(chunk, with_probs=WITH_PROBS, with_gate=WITH_GATE).to_csv(
        output, mode='w' if first else 'a', header=first, index=False)
    first = False
print('predictions written to', output)
display(pd.read_csv(output, nrows=10))
if LABELS_FROM:
    frames = evaluate(os.path.splitext(output)[0] + '_evaluation', bundle=bundle, csv=INPUT_CSV,
                      labels_from=LABELS_FROM, split='csv')
    display(frames['overall'][['origin', 'macro_f1', 'accuracy', 'balanced_accuracy', 'weighted_f1']])